# RaceAI — 統合当日フロー（Layer1 着順 + Layer2 EV）

| 仕様                | パス                                                            |
| ------------------- | --------------------------------------------------------------- |
| 当日予測（Layer 1） | `docs/specs/2026-07-04-today-prediction-design.md`              |
| 統合憲法            | `docs/specs/2026-07-05-var1-integration-architecture-design.md` |

```
Step 0–1   データ取得（JV-Link）
Step 2–6   Layer 1: pure_rank 着順予測 → main/predictions/
Step 7–9   Layer 2: Binary + EV 推奨 → main/results/
```

**Layer 1**: 市場情報を特徴量に使わない（オッズは today_adapter で drop）  
**Layer 2**: init_score(β=0.15) + EV 1.05 + max_picks 2（Phase 1a-A2 凍結）

JV-Link が必要なセルには **⚠️ JV-Link 接続環境が必要** と記載。


In [7]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "common").is_dir() and (cand / "main").is_dir():
            return cand
    raise RuntimeError(f"プロジェクトルートが見つかりません: {start}")


_root = find_project_root(Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("project root:", _root)

project root: C:\Users\syugo\AI\RaceAI_var1.0


In [8]:
%load_ext autoreload
%autoreload 2
from main.notebook_bootstrap import *


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Step 0: 事前準備（当日朝など）

HC/WC（坂路調教・ウッドチップ調教）の当年分を再取得し、前処理を更新する。
当日予測には直近の調教データ（trn*hc*_/trn*wc*_ 12列）が必須だが、
`run_today_se_ra_and_realtime()` は RA/SE + 速報 + オッズのみを取得し
HC/WC はカバーしない（仕様書 2-7節）。

**⚠️ このセルの実行には JV-Link 接続環境が必要です。**


In [9]:
# refresh_today_training_data()

## Step 1: 当日データ取得

`run_today_se_ra_and_realtime()` を実行し、当日（または指定日）の RA/SE を
`main/data/race/race_ra.csv` / `race_se.csv` に保存する。
続けて速報（天候・馬場・馬体重）・0B31 単勝オッズを取得する。

| データ        | Layer 1（着順）                 | Layer 2（EV）               |
| ------------- | ------------------------------- | --------------------------- |
| オッズ        | **drop**（today_adapter）       | **使用**（init_score / EV） |
| 馬体重・DM/TM | 特徴量に反映（HC/WC は Step 0） | binary 特徴量に反映         |

**⚠️ このセルの実行には JV-Link 接続環境が必要です。**


In [10]:
race_day = None  # None = 本日。特定日を指定する場合は "20260704" のように渡す
run_today_se_ra_and_realtime(race_day)

WE v2 snapshot generated: {"date": "20260713", "events": 6, "courses": 3, "events_path": "C:\\Users\\syugo\\AI\\RaceAI_var1.0\\common\\data\\output\\realtime_we_v2\\we_events_20260713_20260713_103023.csv", "snapshot_path": "C:\\Users\\syugo\\AI\\RaceAI_var1.0\\common\\data\\output\\realtime_we_v2\\we_course_snapshot_20260713_20260713_103023.csv"}


CompletedProcess(args=['py', '-3-32', '-c', "import os, sys\nsys.path.insert(0, 'C:\\\\Users\\\\syugo\\\\AI\\\\RaceAI_var1.0')\nos.chdir('C:\\\\Users\\\\syugo\\\\AI\\\\RaceAI_var1.0')\nfrom common.data.src.get_data import run_today_se_ra_and_realtime_merge; run_today_se_ra_and_realtime_merge(dual_pass_se_then_ra=True, target_kubun='both')"], returncode=1, stdout="=== get_race_data: 予測対象レースデータ取得 ===\n取得期間: 20260713000000 -> 20260713235959\nJVOpen From 補正: 20260712000000 (単日指定時の JVOpen -1 回避)\n保存先: C:\\Users\\syugo\\AI\\RaceAI_var1.0\\main\\data\\race\nOption: 1 (通常データ)\n\n--- RACEデータ取得中 (RA, SE) ---\n  [ERR]  JVOpen RACE failed: code=-1 raw=(-1, 0, 0, '')\n!!! Error during data fetch: JVOpen failed with code -1\n", stderr='\nFetching RACE: 0chunks [00:00, ?chunks/s]Traceback (most recent call last):\n  File \x1b"<string>"\x1b, line \x1b4\x1b, in \x1b<module>\x1b\n    from common.data.src.get_data import run_today_se_ra_and_realtime_merge; \x1brun_today_se_ra_and_realtime_merge\x1b\x1b(d

## Step 2: 当日データ → 既存前処理スキーマへの変換

`pure_rank/src/today_adapter.py` の `build_today_merged()` で
`race_ra.csv` / `race_se.csv`（JV-Link 生データ）を
`SE_preprocessed.parquet` / `RA_preprocessed.parquet` と同一スキーマに変換し、
SK（血統）を結合する。オッズ・人気列はこの時点で明示的に drop される。

JV-Link 接続は不要（Step 1 で保存済みの CSV を読むだけ）だが、
Step 1 が未実行の環境では `race_ra.csv` / `race_se.csv` が存在せず
`FileNotFoundError` になる。


In [11]:
cfg = load_config()
race_dir = PROJECT_ROOT / "main" / "data" / "race"
preprocessed_dir = PROJECT_ROOT / cfg["data"]["preprocessed_dir"]

today_merged = build_today_merged(race_dir, preprocessed_dir)
today_merged.head()

  [today_adapter] loaded race_ra.csv: 36 rows, 46 cols
  [today_adapter] loaded race_se.csv: 476 rows, 51 cols
  [today_adapter:RA][WARN] track_condition_code=0 の行が 26 件 （障害 or 馬場状態未確定の可能性）。該当レースはシナリオ生成をスキップすること: ['2026070502010801', '2026070502010802', '2026070502010803', '2026070502010804', '2026070502010805', '2026070502010806', '2026070502010807', '2026070502010808', '2026070502010811', '2026070503020401', '2026070503020402', '2026070503020403', '2026070503020404', '2026070503020405', '2026070503020406', '2026070503020407', '2026070503020411', '2026070503020412', '2026070510020401', '2026070510020402', '2026070510020403', '2026070510020404', '2026070510020405', '2026070510020406', '2026070510020410', '2026070510020412']
  [today_adapter:SE] 市場情報列を drop: ['odds', 'popularity']
  [today_adapter] スキーマ整合性チェック PASS (SE 33列, RA 27列)
  [today_adapter] Merged today data: 476 rows, 44 cols, 36 races


,race_id,year,month_day,course_code,kai,nichi,race_num,wakuban,horse_num,ketto_num,sex_code,age,trainer_code,jockey_code,burden_weight,blinker_code,horse_weight,horse_weight_change,abnormal_code,finish_rank,racetime,time_3f_after,corner_1,corner_2,corner_3,corner_4,hon_shokin,fuka_shokin,running_style_code,mining_predicted_rank,is_win,is_place,grade_code,distance,track_code,horse_count,weather_code,surface_code,track_condition_code,surface_condition,distance_category,race_date,sire_id,bms_id
0,2026070502010801,2026,705,2,1,8,1,1,1,2024100607,2,2,1180,1204,51.0,0,400.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-05,1120002398,1120001954
1,2026070502010801,2026,705,2,1,8,1,2,2,2024100569,1,2,1093,1214,54.0,0,450.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-05,NaN,NaN
2,2026070502010801,2026,705,2,1,8,1,3,3,2024100336,2,2,1051,1091,55.0,0,414.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-05,NaN,NaN
3,2026070502010801,2026,705,2,1,8,1,4,4,2024106853,2,2,1131,1143,55.0,0,418.0,-4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-05,NaN,NaN
4,2026070502010801,2026,705,2,1,8,1,5,5,2024110011,1,2,1180,1197,55.0,0,484.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,0,0,NaN,1200,17,0,1,1,0,10,0,2026-07-05,NaN,NaN


## Step 3–4: 当日特徴量（Layer 1 · 4シナリオ）

`build_today_features()`（bootstrap 経由 = `pure_rank/src/predict_today.py`）で
132列特徴量を `track_condition_code` シナリオ（1=良 … 4=不良）ごとに生成。

JV-Link 不要（Step 2 の `today_merged` + `pure_rank/data/01_preprocessed/` のみ）。


In [12]:
today_features = build_today_features(today_merged, cfg)
{code: df.shape for code, df in today_features.items()}

  SE: 553,887 rows, 33 cols
  RA: 39,570 rows, 27 cols
  SK: 60,648 rows, 3 cols
  [warn] region_code が C:\Users\syugo\AI\RaceAI_var1.0\pure_rank\data\01_preprocessed\SE_preprocessed.parquet に存在しません。transport_category は NaN にフォールバックします（L4復旧確認, 2026-07-10）。
  Merged: 553,887 rows, 47 cols
  Filter applied: 553,887 → 536,776 rows (removed 17,111)


C:\Users\syugo\AI\RaceAI_var1.0\pure_rank\src\predict_today.py:260: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([hist_df, today_placeholder], ignore_index=True, sort=False)


  [build_today_features] Combined: 537,252 rows (hist=536,776, today=476)
  track_code value_counts (rows):
    track_code=10: 4,456
    track_code=11: 72,290
    track_code=12: 11,470
    track_code=17: 113,845
    track_code=18: 46,817
    track_code=20: 24
    track_code=23: 97,669
    track_code=24: 174,325
    track_code=52: 4,646
    track_code=54: 7,863
    track_code=55: 259
    track_code=56: 3,512
    track_code=57: 76
  course_is_small=1 (中間変数): 122,732 rows (22.8%)
  HC: 5,113,087 rows
  WC: 741,428 rows
  [build_today_features] シナリオ間 129 列（132 - 5）の完全一致を確認 (PASS)
  [build_today_features] 特徴量列数=108


{1: (476, 134), 2: (476, 134), 3: (476, 134), 4: (476, 134)}

## Step 5: 推論（アンサンブル予測）

`run_today_predictions()` で 15 モデル（3fold×5seed）アンサンブルにより
`pred_score` / `pred_rank` / `pred_softmax_prob`（参考列）を算出する。
JV-Link 接続不要。


In [13]:
today_with_preds = run_today_predictions(today_features, cfg)
today_with_preds[1].sort_values(["race_id", "pred_rank"]).head(20)

  Loaded: lambdarank_fold1_seed42.txt
  Loaded: lambdarank_fold1_seed43.txt
  Loaded: lambdarank_fold1_seed44.txt
  Loaded: lambdarank_fold1_seed45.txt
  Loaded: lambdarank_fold1_seed46.txt
  Loaded: lambdarank_fold2_seed42.txt
  Loaded: lambdarank_fold2_seed43.txt
  Loaded: lambdarank_fold2_seed44.txt
  Loaded: lambdarank_fold2_seed45.txt
  Loaded: lambdarank_fold2_seed46.txt
  Loaded: lambdarank_fold3_seed42.txt
  Loaded: lambdarank_fold3_seed43.txt
  Loaded: lambdarank_fold3_seed44.txt
  Loaded: lambdarank_fold3_seed45.txt
  Loaded: lambdarank_fold3_seed46.txt
  [run_today_predictions] track_condition_code=1: モデル未使用の特徴量候補列を無視します（安全）: ['hist_last_agari_time_gap', 'hist_turn_surface_win_edge']
  [run_today_predictions] track_condition_code=2: モデル未使用の特徴量候補列を無視します（安全）: ['hist_last_agari_time_gap', 'hist_turn_surface_win_edge']
  [run_today_predictions] track_condition_code=3: モデル未使用の特徴量候補列を無視します（安全）: ['hist_last_agari_time_gap', 'hist_turn_surface_win_edge']
  [run_today_predictions] tr

,race_id,year,month_day,course_code,kai,nichi,race_num,wakuban,horse_num,ketto_num,sex_code,age,trainer_code,jockey_code,burden_weight,blinker_code,horse_weight,horse_weight_change,abnormal_code,finish_rank,racetime,time_3f_after,corner_1,corner_2,corner_3,corner_4,hon_shokin,fuka_shokin,running_style_code,is_win,is_place,grade_code,distance,track_code,horse_count,weather_code,surface_code,distance_category,race_date,sire_id,bms_id,hist_last_rank,hist_avg_rank_3,hist_avg_rank_5,hist_win_rate,hist_place_rate,hist_last_last3f,hist_avg_last3f_3,hist_avg_last3f_5,hist_last_time_dev,hist_avg_time_dev_3,hist_avg_time_dev_5,hist_last_agari_time_gap,hist_same_surface_win_rate,hist_same_course_win_rate,hist_same_dist_win_rate,hist_turn_surface_win_edge,hist_days_since_last,hist_weight_change,hist_total_prize,hist_avg_prize_3,hist_same_weather_win_rate,hist_same_weather_avg_rank,hist_same_course_dist_win_rate,hist_same_grade_win_rate,hist_top_grade_exp_count,hist_exact_dist_win_rate,hist_best_grade_ever,hist_grade_diff,hist_avg_rank_top_grade,hist_front_running_pref,season_sex_score,wakuban_surface,front_pref_x_small,field_avg_win_rate,field_avg_prize,win_rate_vs_field,prize_vs_field,hist_sire_win_rate_ts,hist_sire_surface_win_rate_ts,hist_sire_dist_win_rate_ts,hist_sire_dist_diff,hist_bms_win_rate_ts,hist_jockey_win_rate_cum,hist_jockey_win_rate_30d,hist_jockey_place_rate_30d,hist_jockey_win_rate_60d,hist_jockey_course_win_rate,hist_trainer_win_rate_cum,hist_trainer_win_rate_30d,hist_trainer_win_rate_60d,hist_trainer_surface_win_rate,hist_jockey_surface_win_rate_ts,hist_trainer_course_win_rate_ts,hist_speed_idx_last,hist_speed_idx_best,hist_speed_idx_avg3,hist_speed_idx_cond_best,field_z_time_dev,field_z_prize,field_z_last3f,field_z_win_rate,field_z_speed_idx,field_z_place_rate,field_front_runner_density,relative_post_position,trn_hc_last_3f_sec,trn_hc_last_4f_sec,trn_hc_last_200_sec,trn_hc_best_3f_14d,trn_hc_best_200_14d,trn_hc_count_14d,trn_wc_last_3f_sec,trn_wc_last_4f_sec,trn_wc_last_1f_sec,trn_wc_best_3f_14d,trn_wc_best_1f_14d,trn_wc_count_14d,trn_total_count_14d,trn_hc_rank_3f,trn_hc_rank_200,trn_hc_zscore_3f,trn_wc_rank_3f,trn_wc_zscore_3f,trn_hc_3f_delta,trn_hc_200_delta,trn_wc_3f_delta,trn_count_delta,hist_same_condition_win_rate,hist_surface_condition_win_rate,hist_best_time_same_cond,track_condition_code,surface_condition,lr_label,pred_score,pred_rank,pred_softmax_prob
475,2026070502010801,2026,705,2,1,8,1,5,5,2024110011,1,2,1180,1197,55.0,0,484.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,1200,17,0,1,1,0,2026-07-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.998186,5.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.085199,NaN,NaN,0.038462,0.125364,0.120346,NaN,0.266667,0.112648,0.096090,0.194444,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,inf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,11,NaN,-0.071406,1,0.285642
398,2026070502010801,2026,705,2,1,8,1,6,6,2024100652,1,2,1119,1170,55.0,0,486.0,-2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,1200,17,0,1,1,0,2026-07-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.998186,6.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.123141,NaN,NaN,0.161290,0.148501,0.049367,NaN,0.000000,0.050487,0.125193,0.059524,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,inf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,11,NaN,-0.534602,2,0.155286
397,2026070502010801,2026,705,2,1,8,1,1,1,2024100607,2,2,1180,1204,51.0,0,400.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,1200,17,0,1,1,0,2026-07-05,1120002398,1120001954,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.998186,1.0,NaN,0.0,NaN,NaN,NaN,0.080677,0.079523,0.0812

## Step 6: 出力 — Layer 1 着順 CSV

```
main/predictions/{YYYYMMDD}/{開催場所名}/{良|稍重|重|不良}/
    race_{race_id}_pred.csv
    summary.csv
```

4シナリオすべてを出力（馬場発表前の what-if 用）。Layer 2 の inline z には
**実馬場シナリオ 1 本**（Step 7 で `track_condition_code` 指定）を使う。

出力前に市場情報混入チェック（`FORBIDDEN_MARKET_COLS`）を実行。JV-Link 不要。


In [14]:
out_root = PROJECT_ROOT / "main" / "predictions"
write_predictions(today_with_preds, out_root)

  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\函館\良: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\福島\良: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\小倉\良: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\函館\稍重: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\福島\稍重: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\小倉\稍重: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\函館\重: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\福島\重: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\小倉\重: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI_var1.0\main\predictions\20260705\函館\不良: 12 races
  [write_predictions] C:\Users\syugo\AI\RaceAI

## Step 7–9: Layer 2 — Binary + EV（**Var1.0 必須**）

**Var1.0（pure_rank）が無いと Layer 2 は意味を持ちません。**

| 段階     | Var1.0 の役割                                                                  |
| -------- | ------------------------------------------------------------------------------ |
| Step 5–6 | `pure_rank/models` LambdaRank **v39_course_slim** → `pred_score` / `pred_rank` |
| Step 8   | `pred_score` → レース内 z-score → **`var1_pure_score_z`**                      |
| Step 9   | Binary **init_score** = `market_log_odds + 0.15 × var1_pure_score_z`           |

Step 7 では必ず `rank_preds=today_with_preds`（Step 5 の結果）を渡すこと。

**出力先（Step 7 実行後）**

| 用途                         | パス                                                       |
| ---------------------------- | ---------------------------------------------------------- |
| Var1 着順（レース別）        | `main/predictions/{日付}/{競馬場}/{馬場}/race_*_pred.csv`  |
| EV 推奨（全頭）              | `main/results/today_recommendations_*.csv`                 |
| **競馬場×馬場 what-if 表示** | `main/results/{競馬場}/馬場_{良\|稍重\|重\|不良}/{n}R.csv` |
| 全頭 wide（4馬場）           | `main/results/today_predictions_with_bets.csv`             |

`馬場_*/*.csv` の **予測スコア=Var1 softmax**、**モデル勝率シェア=Layer2 最終勝率**、**期待値=EV**。


In [15]:
# --- Layer 2 パラメータ（必要に応じて変更）---
TRACK_CONDITION = None  # None=RA から自動。1=良, 2=稍重, 3=重, 4=不良
ODDS_CSV = PROJECT_ROOT / "common" / "data" / "output" / "realtime_odds" / "o1_odds.csv"
BANKROLL = 100_000

summary = run_unified_today(
    rank_features=today_features,
    rank_preds=today_with_preds,  # ← Step 5 の Var1.0 LambdaRank 結果（必須）
    track_condition_code=TRACK_CONDITION,
    odds_csv=ODDS_CSV if ODDS_CSV.exists() else None,
    bankroll=BANKROLL,
    write_rank=False,
    write_bets=True,
)
tc = summary["track_condition_code"]
print(
    f"[Var1→Layer2] track_condition={tc} | "
    f"models={cfg['data']['features_version']} @ {cfg['data']['models_dir']} | "
    f"init_score beta=0.15 | "
    f"venue CSV={summary.get('venue_export_files', 0)} files → {summary['results_dir']}"
)
summary

  [today_adapter] loaded race_ra.csv: 36 rows, 46 cols
  [today_adapter] loaded race_se.csv: 476 rows, 51 cols
  [today_adapter:RA][WARN] track_condition_code=0 の行が 26 件 （障害 or 馬場状態未確定の可能性）。該当レースはシナリオ生成をスキップすること: ['2026070502010801', '2026070502010802', '2026070502010803', '2026070502010804', '2026070502010805', '2026070502010806', '2026070502010807', '2026070502010808', '2026070502010811', '2026070503020401', '2026070503020402', '2026070503020403', '2026070503020404', '2026070503020405', '2026070503020406', '2026070503020407', '2026070503020411', '2026070503020412', '2026070510020401', '2026070510020402', '2026070510020403', '2026070510020404', '2026070510020405', '2026070510020406', '2026070510020410', '2026070510020412']
  [today_adapter:SE] 市場情報列を drop: ['odds', 'popularity']
  [today_adapter] スキーマ整合性チェック PASS (SE 33列, RA 27列)
  [today_adapter] Merged today data: 476 rows, 44 cols, 36 races
本推奨は市場に対する相対的な損失最小化を目的とし、黒字化を保証するものではない（fold2 OOS実測: ROI 81.89%、元本の約18%の期待損失）
[Var1→Layer2] 

{'track_condition_code': 1,
 'rank_scenarios': [1, 2, 3, 4],
 'fusion_alpha': 1.05562682657624,
 'fusion_beta': 0.3494051978595059,
 'mode': 'loss_min_top1',
 'odds_source': 'realtime_o1',
 'recommendations': 29,
 'skipped_races': 7,
 'venue_export_files': 144,
 'results_dir': 'C:\\Users\\syugo\\AI\\RaceAI_var1.0\\main\\results',
 'predictions_with_bets_csv': 'C:\\Users\\syugo\\AI\\RaceAI_var1.0\\main\\results\\today_predictions_with_bets.csv'}

## Step 9 確認: 推奨リスト表示

`race_id` から **開催日・競馬場・R** を復元して表示（例: `函館 5R`）。
全列は `main/results/today_recommendations_*.csv` を参照。


In [16]:
from datetime import datetime

COURSE_NAMES = {
    1: "札幌",
    2: "函館",
    3: "福島",
    4: "新潟",
    5: "東京",
    6: "中山",
    7: "中京",
    8: "京都",
    9: "阪神",
    10: "小倉",
}


def _add_race_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "race_id" not in out.columns:
        return out
    rid = out["race_id"].astype(str).str.zfill(16)
    if "month_day" not in out.columns:
        md = rid.str[4:8]
        out["開催日"] = md.str[:2] + "/" + md.str[2:4]
    if "course_code" not in out.columns:
        cc = pd.to_numeric(rid.str[8:10], errors="coerce")
        out["競馬場"] = cc.map(COURSE_NAMES).fillna(
            cc.astype("Int64").astype(str) + "場"
        )
    elif "競馬場" not in out.columns:
        out["競馬場"] = pd.to_numeric(out["course_code"], errors="coerce").map(
            COURSE_NAMES
        )
    if "race_num" not in out.columns:
        rn = pd.to_numeric(rid.str[14:16], errors="coerce")
        out["R"] = rn.astype("Int64").astype(str) + "R"
    elif "R" not in out.columns:
        out["R"] = (
            pd.to_numeric(out["race_num"], errors="coerce").astype("Int64").astype(str)
            + "R"
        )
    out["レース"] = out["競馬場"].astype(str) + " " + out["R"].astype(str)
    return out


results_dir = PROJECT_ROOT / "main" / "results"
date_str = datetime.now().strftime("%Y%m%d")
rec_csv = results_dir / f"today_recommendations_{date_str}.csv"
if not rec_csv.exists():
    rec_csv = results_dir / "today_recommendations.csv"

if rec_csv.exists():
    recs = pd.read_csv(rec_csv)
    if "is_recommended" in recs.columns and recs["is_recommended"].dtype == object:
        recs["is_recommended"] = (
            recs["is_recommended"].astype(str).str.lower().eq("true")
        )
    ev_col = "ev_rate" if "ev_rate" in recs.columns else "expected_value"

    # Step 5 Var1.0 着順を結合（Layer2 が Var1 を使っていることの確認用）
    tc = summary.get("track_condition_code", 1) if "summary" in dir() else 1
    var1_rank = today_with_preds[tc][
        ["race_id", "horse_num", "pred_score", "pred_rank"]
    ].copy()
    var1_rank["race_id"] = var1_rank["race_id"].astype(str)
    var1_rank = var1_rank.rename(
        columns={"pred_score": "var1_pred_score", "pred_rank": "var1_pred_rank"}
    )

    picks = _add_race_labels(recs[recs["is_recommended"] == True])  # noqa: E712
    if "race_id" in picks.columns:
        picks["race_id"] = picks["race_id"].astype(str)
    if "horse_num" in picks.columns:
        picks["horse_num"] = pd.to_numeric(picks["horse_num"], errors="coerce")
        var1_rank["horse_num"] = pd.to_numeric(var1_rank["horse_num"], errors="coerce")
        picks = picks.merge(var1_rank, on=["race_id", "horse_num"], how="left")

    cols = [
        c
        for c in [
            "開催日",
            "競馬場",
            "R",
            "レース",
            "horse_num",
            "var1_pred_rank",
            "var1_pred_score",
            "var1_pure_score_z",
            "odds",
            "model_prob",
            ev_col,
            "kelly_bet_yen",
            "race_id",
        ]
        if c in picks.columns
    ]
    sort_keys = [c for c in ["競馬場", "R", ev_col] if c in picks.columns]
    if sort_keys:
        picks = picks.sort_values(
            sort_keys, ascending=[True, True, False][: len(sort_keys)]
        )
    rename = {
        "horse_num": "馬番",
        "var1_pred_rank": "Var1予想着順",
        "var1_pred_score": "Var1スコア",
        "var1_pure_score_z": "Var1_z(init_score用)",
        "odds": "オッズ",
        "model_prob": "モデル勝率",
        ev_col: "期待値",
        "kelly_bet_yen": "推奨額",
        "race_id": "race_id(参考)",
    }
    view = picks[cols].rename(
        columns={k: v for k, v in rename.items() if k in picks.columns}
    )
    print(f"推奨 {len(view)} 件 / 全 {len(recs)} 頭 → {rec_csv}")
    if "var1_pure_score_z" not in recs.columns:
        print("[WARN] var1_pure_score_z なし — Step 7 を rank_preds 付きで再実行")
    display(view)
else:
    print(f"推奨 CSV が見つかりません: {results_dir}")

KeyError: 'is_recommended'

## 市場情報混入チェック（Layer 1 のみ）

Layer 1（`pure_rank/src`）にオッズ・init_score が**特徴量として**混入していないことを確認する。
Layer 2（`model_training` / `strategy`）は init_score / EV でオッズを使うのが正常。

```bash
grep -rn "odds\|popularity\|ninki\|market_log_odds\|init_score" pure_rank/src/ --include="*.py"
```


In [ ]:
import re

pure_rank_src = PROJECT_ROOT / "pure_rank" / "src"
pattern = re.compile(r"\b(odds|popularity|ninki|market_log_odds|init_score)\b")
allow_substrings = (
    "FORBIDDEN_MARKET_COLS",
    "simulate_ev",
    "build_win_odds",
    "market_win_probs",
    "ベッティング",
    "betting layer",
    "評価専用",
    "drop",
    "禁止",
    "exclude",
)
hits = []
for py in sorted(pure_rank_src.glob("*.py")):
    for i, line in enumerate(py.read_text(encoding="utf-8").splitlines(), 1):
        if not pattern.search(line):
            continue
        if any(a in line for a in allow_substrings):
            continue
        hits.append(f"{py.name}:{i}: {line.strip()[:100]}")
print(f"Layer 1 要確認ヒット: {len(hits)} 行")
for h in hits[:20]:
    print(h)
if len(hits) > 20:
    print(f"... 他 {len(hits) - 20} 行")

Layer 1 要確認ヒット: 58 行
common.py:39: "odds", "popularity", "win_odds", "place_odds",
common.py:40: "quinella_odds", "market_prob", "market_log_odds",
common.py:41: "init_score", "ninki",
create_features.py:9: - オッズ・人気 (odds, popularity) を特徴量に含めない
create_features.py:10: - market_log_odds / init_score を使わない
predict.py:368: odds = np.asarray(odds, dtype=float)
predict.py:369: if len(odds) < 2:
predict.py:371: valid = odds > 0
predict.py:374: raw = np.where(valid, 1.0 / np.clip(odds, 1.01, None), 0.0)
predict.py:404: se = pd.read_parquet(se_path, columns=["race_id", "horse_num", "odds"])
predict.py:405: se = se[se["odds"] > 0]
predict.py:409: zip(grp["horse_num"].astype(int), grp["odds"].astype(float))
predict.py:419: """(scores, is_win, odds) のレース単位リスト。オッズ欠損レースは除外。"""
predict.py:524: print(f"  Win odds races: {len(win_odds_lookup):,}")
predict.py:532: print(f"  Usable races with win odds: {len(races):,}")
predict.py:1326: WideOdds_{year}.csv を複数年読み込み、race_id -> {(h1,h2): odds} を返す。
predict.

## （参考）Layer 1 回帰テスト

JV-Link 不要。コード変更後の健全性確認用。

```bash
python pure_rank/src/regression_test_today.py
```


In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "pure_rank" / "src" / "regression_test_today.py"),
    ],
    cwd=PROJECT_ROOT,
    check=False,
)

CompletedProcess(args=['c:\\Users\\syugo\\anaconda3\\python.exe', 'C:\\Users\\syugo\\AI\\RaceAI_var1.0\\pure_rank\\src\\regression_test_today.py'], returncode=1)